# Demo 2 - O modelo funciona fora do laboratório? Testando com a base PhiUSIIL

### Cybersecurity MBA - Artificial Intelligence & Robotics Trends (SEGR)

### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

---

**Continuação da Demo 1.** Lá, treinamos um detector de URLs maliciosas com o Kaggle URL Dataset e chegamos a mais de 95% de recall. Agora vamos testá-lo contra o mundo real.

Vamos percorrer 5 etapas:

0. Preparar o ambiente e **recriar rapidamente** o modelo da Demo 1
1. Conhecer a base **PhiUSIIL**, de phishing recente
2. Fazer a **prova de fogo**: o modelo funciona fora da base em que foi treinado?
3. Testar URLs ao vivo e escolher o **limiar**
4. Tentar **enganar** o modelo (ataque adversarial)

## Vocabulário rápido (volte aqui sempre que precisar)

| Termo | O que significa | Analogia |
|---|---|---|
| **Dataset** | Tabela com exemplos já conhecidos | Um caderno de casos antigos |
| **Rótulo (label)** | A resposta certa de cada exemplo: malicioso ou legítimo | O gabarito |
| **Treino** | Parte dos dados que o modelo vê para aprender | Os exercícios que o aluno estuda |
| **Teste** | Parte dos dados que o modelo **nunca viu**, usada para avaliar | A prova final |
| **Modelo** | O "cérebro" que resulta do aprendizado | O aluno depois de estudar |
| **Falso negativo** | Ataque real que o modelo chamou de legítimo | O alarme que **não tocou** com o ladrão dentro |
| **Falso positivo** | Site legítimo que o modelo chamou de malicioso | O alarme que tocou à toa |
| **Recall** | Dos ataques reais, quantos o modelo pegou | Quantos ladrões o alarme detectou |
| **Precision** | Dos alertas que o modelo deu, quantos eram ataque de verdade | Quantos toques de alarme eram ladrão de fato |

## Etapa 0 - Preparando o ambiente e recriando o modelo da Demo 1

In [ ]:
import os, re, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                             confusion_matrix, ConfusionMatrixDisplay)

pd.set_option("display.max_colwidth", 90)
print("Bibliotecas carregadas.")

### Baixando as duas bases de dados

| Base | Ano | Origem | Tamanho |
|---|---|---|---|
| **Kaggle URL Dataset** (Demo 1) | ~2015 | Kaggle (`antonyj453/urldataset`), cópia no repositório da disciplina | ~420 mil URLs |
| **PhiUSIIL** | 2024 | UCI Machine Learning Repository (licença CC BY 4.0) | ~235 mil URLs |

> Referência da base PhiUSIIL: Prasad, A. & Chandra, S. (2024). *PhiUSIIL: A diverse security profile empowered phishing URL detection framework based on similarity index and incremental learning.* Computers & Security.

In [ ]:
URL_KAGGLE = "https://raw.githubusercontent.com/ahirtonlopes/AI-and-Robotics-Trends/main/Datasets/url_data.csv"
URL_PHIUSIIL = "https://archive.ics.uci.edu/static/public/967/phiusiil+phishing+url+dataset.zip"

if not os.path.exists("url_data.csv"):
    urllib.request.urlretrieve(URL_KAGGLE, "url_data.csv")

if not os.path.exists("PhiUSIIL_Phishing_URL_Dataset.csv"):
    urllib.request.urlretrieve(URL_PHIUSIIL, "phiusiil.zip")
    zipfile.ZipFile("phiusiil.zip").extractall(".")

print("Arquivos prontos:", [f for f in os.listdir(".") if f.endswith(".csv")])

### Recriando o que fizemos na Demo 1

São as mesmas funções e o mesmo modelo final (palavras em pedaços de letras, pesos balanceados). Se você acabou de rodar a Demo 1 no mesmo ambiente, é uma repetição rápida.

In [ ]:
def limpar_url(u):
    u = re.sub(r"\s+", "", str(u).lower())      # remove espaços e quebras de linha
    u = re.sub(r"^https?://", "", u)
    u = re.sub(r"^www\.", "", u)
    return u.rstrip("/")

# exemplo
print(limpar_url("HTTPS://www.Banco-Seguro.com/Login/"))

In [ ]:
kaggle_bruta = pd.read_csv("url_data.csv")
kaggle = pd.DataFrame({
    "url": kaggle_bruta["url"].map(limpar_url),
    "malicioso": (kaggle_bruta["label"] == "bad").astype(int),
}).drop_duplicates("url")
print(f"Base Kaggle: {len(kaggle):,} URLs")

In [ ]:
def treinar(df, analisador="word", balanceado=False, seed=42):
    """Treina um modelo e devolve tudo que precisamos para avaliar."""
    treino, teste = train_test_split(df, test_size=0.2, random_state=seed, stratify=df["malicioso"])

    if analisador == "word":      # olha palavras inteiras
        vec = TfidfVectorizer(sublinear_tf=True)
    else:                         # olha pedaços de 2 a 4 letras
        vec = TfidfVectorizer(analyzer="char", ngram_range=(2, 4), max_features=300000, sublinear_tf=True)

    X_treino = vec.fit_transform(treino["url"])
    X_teste = vec.transform(teste["url"])

    modelo = LogisticRegression(max_iter=1000, class_weight="balanced" if balanceado else None)
    modelo.fit(X_treino, treino["malicioso"])

    return {"modelo": modelo, "vec": vec, "teste": teste, "X_teste": X_teste,
            "pred": modelo.predict(X_teste)}


def metricas(y_real, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred).ravel()
    return {"acurácia": accuracy_score(y_real, y_pred),
            "recall": recall_score(y_real, y_pred),
            "precision": precision_score(y_real, y_pred),
            "ataques que passaram (FN)": fn,
            "alarmes falsos (FP)": fp}

In [ ]:
r_B = treinar(kaggle, analisador="char", balanceado=True)     # o melhor modelo da Demo 1
y_real = r_B["teste"]["malicioso"]

m_kaggle = metricas(y_real, r_B["pred"])
print(f"Modelo da Demo 1 na própria base Kaggle: acurácia {m_kaggle['acurácia']:.1%} | recall {m_kaggle['recall']:.1%}")

## Etapa 1 - A base PhiUSIIL: phishing recente

Phishing evoluiu. Hoje há domínios falsos de bancos, marcas conhecidas e páginas hospedadas em serviços legítimos. Vamos conhecer a base PhiUSIIL.

### Atenção ao rótulo invertido!

Essa base usa a convenção **oposta** à nossa: `label = 1` significa **legítimo** e `label = 0` significa **phishing**. Se não percebermos isso, o modelo inteiro fica de cabeça para baixo. Vamos converter para o nosso padrão (`malicioso = 1`).

In [ ]:
phiusiil_bruta = pd.read_csv("PhiUSIIL_Phishing_URL_Dataset.csv", usecols=["URL", "label"])

phiusiil = pd.DataFrame({
    "url": phiusiil_bruta["URL"].map(limpar_url),
    "malicioso": (phiusiil_bruta["label"] == 0).astype(int),   # na base original: 0 = phishing
}).drop_duplicates("url")

print(f"Base PhiUSIIL: {len(phiusiil):,} URLs")
phiusiil.sample(8, random_state=7)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for a, (nome, df) in zip(ax, [("Base Kaggle", kaggle), ("Base PhiUSIIL", phiusiil)]):
    contagem = df["malicioso"].value_counts().sort_index()
    a.bar(["legítimo", "malicioso"], contagem.values, color=["#4c9a6a", "#c0504d"])
    a.set_title(f"{nome}\n{contagem[1] / len(df):.0%} maliciosas")
    a.set_ylabel("URLs")
plt.tight_layout(); plt.show()

Vamos repetir **exatamente o mesmo processo** da Demo 1, agora treinando e testando na base PhiUSIIL.

In [ ]:
r_phi = treinar(phiusiil, analisador="char", balanceado=True)
y_phi = r_phi["teste"]["malicioso"]

m_phi = metricas(y_phi, r_phi["pred"])
pd.DataFrame({"Base PhiUSIIL": m_phi}).T.style.format(
    {"acurácia": "{:.1%}", "recall": "{:.1%}", "precision": "{:.1%}",
     "ataques que passaram (FN)": "{:,.0f}", "alarmes falsos (FP)": "{:,.0f}"})

O desempenho é um pouco menor que na base Kaggle: o phishing moderno é **mais difícil** de identificar só pelo texto da URL.

## Etapa 2 - A prova de fogo

Até aqui, cada modelo foi testado com dados **da mesma base** em que treinou. Mas na vida real ele vai encontrar URLs de **outra origem, outra época, outro perfil**.

Pergunta: **o modelo treinado na base Kaggle funciona na base PhiUSIIL? E vice-versa?**

In [ ]:
def testar_em(resultado, df_teste, vec_do_modelo):
    X = vec_do_modelo.transform(df_teste["url"])
    pred = resultado["modelo"].predict(X)
    return metricas(df_teste["malicioso"], pred)

cruzado = pd.DataFrame({
    "Treinou Kaggle -> testou Kaggle": metricas(y_real, r_B["pred"]),
    "Treinou PhiUSIIL -> testou PhiUSIIL":     m_phi,
    "Treinou Kaggle -> testou PHIUSIIL":   testar_em(r_B, r_phi["teste"], r_B["vec"]),
    "Treinou PhiUSIIL -> testou KAGGLE":   testar_em(r_phi, r_B["teste"], r_phi["vec"]),
}).T

cruzado[["acurácia", "recall", "precision"]].style.format("{:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
cores = ["#4c9a6a", "#4c9a6a", "#c0504d", "#c0504d"]
ax.barh(cruzado.index[::-1], cruzado["acurácia"][::-1], color=cores[::-1])
ax.set_xlim(0, 1); ax.set_xlabel("Acurácia")
ax.set_title("Mesmo modelo, dados de outra origem")
for i, v in enumerate(cruzado["acurácia"][::-1]):
    ax.text(v + 0.01, i, f"{v:.0%}", va="center")
plt.tight_layout(); plt.show()

### O que aconteceu?

Quando o modelo sai da base em que treinou, o desempenho **desaba**. Ele não aprendeu "o que é phishing". Aprendeu **o jeito particular de cada base**. Vamos olhar um exemplo concreto: a presença de um caminho depois do domínio (`site.com/pasta/pagina`).

In [ ]:
def tem_caminho(df):
    return df["url"].str.contains("/").groupby(df["malicioso"]).mean().rename({0: "legítimas", 1: "maliciosas"})

comparacao = pd.DataFrame({"Base Kaggle": tem_caminho(kaggle), "Base PhiUSIIL": tem_caminho(phiusiil)})
print("Proporção de URLs que têm '/' (um caminho depois do domínio):")
comparacao.style.format("{:.0%}")

Na base PhiUSIIL, **nenhuma URL legítima tem caminho**: elas são só a página inicial de cada site (`universidade.edu`). Já na Kaggle, quase todas as legítimas têm caminho (`site.com/pagina`). Cada base foi **coletada de um jeito**, e o modelo aprende esse "atalho" em vez do comportamento malicioso.

Isso se chama **viés de coleta** (ou *dataset shift*), e é um dos maiores riscos de ML em segurança: o modelo vai muito bem no laboratório e falha em produção.

> **Lição 3:** nota alta no teste não prova que o modelo funciona no mundo real. Teste com dados de **outra origem**.

### Como reduzir o problema?

Treinar com **dados de várias origens**. Vamos combinar as duas bases e avaliar de novo, separando o resultado por origem.

In [ ]:
combinada = pd.concat([kaggle.assign(fonte="kaggle"), phiusiil.assign(fonte="phiusiil")]).drop_duplicates("url")
estrato = combinada["fonte"] + combinada["malicioso"].astype(str)

treino_c, teste_c = train_test_split(combinada, test_size=0.2, random_state=42, stratify=estrato)

vec_final = TfidfVectorizer(analyzer="char", ngram_range=(2, 4), max_features=300000, sublinear_tf=True)
X_treino_c = vec_final.fit_transform(treino_c["url"])

modelo_final = LogisticRegression(max_iter=1000, class_weight="balanced")
modelo_final.fit(X_treino_c, treino_c["malicioso"])

linhas = {}
for fonte in ["kaggle", "phiusiil"]:
    t = teste_c[teste_c["fonte"] == fonte]
    linhas[f"Modelo combinado -> {fonte}"] = metricas(t["malicioso"], modelo_final.predict(vec_final.transform(t["url"])))

pd.DataFrame(linhas).T.style.format(
    {"acurácia": "{:.1%}", "recall": "{:.1%}", "precision": "{:.1%}",
     "ataques que passaram (FN)": "{:,.0f}", "alarmes falsos (FP)": "{:,.0f}"})

Agora o **mesmo modelo** funciona razoavelmente nas duas origens. Ele é um pouco pior que o "especialista" treinado só na base PhiUSIIL, mas é muito mais robusto fora do ambiente de treino. Em segurança, robustez costuma valer mais que uma nota alta em condições ideais.

## Etapa 3 - Teste ao vivo

O modelo devolve uma **probabilidade de risco** entre 0 e 1. Quem decide o que fazer com ela é o **limiar** (threshold): acima dele, a URL é bloqueada.

- Limiar **baixo** (ex.: 0,3): bloqueia mais, pega mais ataques, mas gera mais alarmes falsos.
- Limiar **alto** (ex.: 0,9): bloqueia menos, incomoda menos o usuário, mas deixa passar mais ataques.

In [ ]:
def risco(url):
    """Probabilidade (0 a 1) de a URL ser maliciosa, segundo o modelo."""
    return modelo_final.predict_proba(vec_final.transform([limpar_url(url)]))[0, 1]

LIMIAR = 0.90   # <- experimente mudar: 0.3, 0.5, 0.7, 0.9, 0.99

exemplos = [
    # (URL, realidade)
    ("google.com", "legítima"),
    ("github.com/ahirtonlopes", "legítima"),
    ("fiap.com.br", "legítima"),
    ("nubank.com.br", "legítima"),
    ("en.wikipedia.org/wiki/Phishing", "legítima"),
    ("paypa1-secure-login.com/verify/account", "PHISHING"),
    ("secure-itau.com.br-atualizacao.top/login.php", "PHISHING"),
    ("accounts.google.com.verify-session.ru/signin", "PHISHING"),
    ("microsoft-365-support.xyz/password-reset", "PHISHING"),
    ("192.168.4.7/admin/login.php", "PHISHING"),
    ("nubank-seguranca.info/app/confirmar", "PHISHING"),
]

linhas = []
for url, real in exemplos:
    p = risco(url)
    veredito = "BLOQUEAR" if p >= LIMIAR else "liberar"
    certo = (veredito == "BLOQUEAR") == (real == "PHISHING")
    linhas.append({"URL": url, "realidade": real, "risco": round(p, 2), "decisão": veredito, "acertou?": "sim" if certo else "NÃO"})

pd.DataFrame(linhas)

> Atenção: 11 exemplos escolhidos à mão **não** são uma avaliação do modelo. São só uma ilustração. A avaliação de verdade é a das tabelas anteriores, com milhares de URLs.

### O efeito do limiar no conjunto de teste inteiro

In [ ]:
prob_teste = modelo_final.predict_proba(vec_final.transform(teste_c["url"]))[:, 1]
y_teste = teste_c["malicioso"].values

linhas = []
for lim in [0.3, 0.5, 0.7, 0.9, 0.99]:
    pred = (prob_teste >= lim).astype(int)
    m = metricas(y_teste, pred)
    linhas.append({"limiar": lim, "recall": m["recall"], "precision": m["precision"],
                   "ataques que passaram": m["ataques que passaram (FN)"],
                   "alarmes falsos": m["alarmes falsos (FP)"]})

pd.DataFrame(linhas).set_index("limiar").style.format(
    {"recall": "{:.1%}", "precision": "{:.1%}", "ataques que passaram": "{:,.0f}", "alarmes falsos": "{:,.0f}"})

Repare: com o limiar de 0,90 os 11 exemplos ao vivo deram todos certo, mas no conjunto de teste inteiro esse mesmo limiar **deixa passar cerca de 30% dos ataques**. É por isso que olhamos milhares de casos, e não só uma amostra bonita. Volte à célula dos exemplos, troque `LIMIAR` para 0,5 e veja `google.com` e `github.com` passarem a ser bloqueados.

Nenhum limiar é "o certo". Um banco pode preferir limiar baixo (qualquer suspeita é investigada). Um provedor de e-mail com milhões de usuários pode preferir um mais alto, para não bloquear mensagens legítimas.

### Agora é com você

Edite a variável abaixo com uma URL (real ou inventada) e rode a célula.

In [ ]:
minha_url = "bancodobrasil-atualizacao-cadastral.com/login"

p = risco(minha_url)
print(f"Risco estimado: {p:.0%}")
print("Decisão:", "BLOQUEAR" if p >= LIMIAR else "liberar", f"(limiar = {LIMIAR})")

## Etapa 4 - Tentando enganar o modelo

Um atacante que sabe como o modelo funciona pode **alterar a URL** para baixar a pontuação de risco, sem mudar o que a página faz. Vamos simular isso de forma simples: pegar uma URL de phishing e **acrescentar pedaços "de aparência legítima"**, escolhendo a cada passo o que mais reduz o risco.

In [ ]:
# Segmentos de caminho mais comuns em URLs legítimas do treino
legitimas = treino_c[treino_c["malicioso"] == 0]["url"]
segmentos = (legitimas.str.split("/").explode().str.strip()
             .loc[lambda s: s.str.len().between(3, 12) & ~s.str.contains(r"\.|\?|=")]
             .value_counts().head(150).index.tolist())

def ataque_guloso(url, passos=6):
    atual = limpar_url(url)
    historico = [(atual, risco(atual))]
    for _ in range(passos):
        candidatos = [atual + "/" + s for s in segmentos]
        probs = modelo_final.predict_proba(vec_final.transform(candidatos))[:, 1]
        melhor = int(np.argmin(probs))
        if probs[melhor] >= historico[-1][1]:
            break
        atual = candidatos[melhor]
        historico.append((atual, probs[melhor]))
    return historico

for url, p in ataque_guloso("paypa1-secure-login.com/verify/account"):
    print(f"{p:5.0%}  {url}")

Se o risco caiu, o atacante acabou de **evadir** o detector sem tocar na página maliciosa. Isso é um **ataque adversarial**, e é o tema de uma das próximas aulas: modelos de ML também são alvos.

## O que aprendemos nas duas demos

1. **Acurácia engana.** Em segurança, olhe recall, precision e a matriz de confusão. *(Demo 1)*
2. **Melhorar é escolher.** Mais recall custa mais alarmes falsos. O limiar é uma decisão de negócio e de risco. *(Demo 1 e 2)*
3. **Dados importam mais que o algoritmo.** Bases desbalanceadas ou com viés de coleta geram modelos que brilham no laboratório e falham em produção. *(Demo 2)*
4. **Teste em dados de outra origem** antes de confiar no modelo. *(Demo 2)*
5. **O modelo também é uma superfície de ataque.** Quem entende como ele decide consegue enganá-lo. *(Demo 2)*

### Próximos passos
- **Aula 2:** vamos aprofundar métricas e aplicar aprendizado supervisionado a outros problemas de segurança (fraude e detecção de intrusão).
- **Aula 3:** vamos ver ataques adversariais e como proteger modelos.